# Unit economics của một đơn hàng: đơn nhỏ có lỗ không, và tăng giá 5% được phép mất bao nhiêu đơn?

Bộ `online-retail-ii`, sheet **`Year 2010-2011`** (541 910 dòng, 01/12/2010 tới 09/12/2011): cửa hàng quà tặng
online ở Anh, bán cho cả khách lẻ lẫn khách sỉ. Quy tắc dùng chung của track: bỏ hoá đơn bắt đầu bằng `C`, bỏ
dòng `Quantity <= 0` hoặc `Price <= 0`; doanh thu dòng = `Quantity * Price` (GBP, doanh thu gộp).

**Bối cảnh.** CFO hỏi hai câu trước buổi họp ngân sách: "Đơn nhỏ có đang lỗ không, có nên đặt giá trị đơn tối
thiểu?" và "Nếu tăng giá 5%, mình được phép mất bao nhiêu đơn mà lợi nhuận không giảm?". Bộ dữ liệu chỉ có doanh
thu, **không có giá vốn hay chi phí giao hàng**. Bạn dùng một bộ giả định chi phí ghi rõ là giả định, và cho CFO
thấy kết luận nhạy với giả định nào.

**Giả định chi phí (ví dụ giả định, không phải số của doanh nghiệp này):**

| Khoản | Giả định | Ghi chú |
| --- | --- | --- |
| Giá vốn hàng bán (COGS) | 60% doanh thu ở giá hiện tại | Quà tặng bán sỉ, gross margin khoảng 40% |
| Phí thanh toán | 2% doanh thu | Thẻ / cổng thanh toán, tỉ lệ theo số tiền thu |
| Chi phí xử lý mỗi đơn | 8.0 GBP | Đóng gói, giao vận, chăm sóc khách |
| Chi phí mỗi dòng hàng | 0.25 GBP | Nhặt hàng trong kho |

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm hoặc hằng. Điền vào chỗ `# TODO`; hàm không được sửa dữ liệu đầu vào.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu thật và vẽ; khi chấm chúng bị bỏ qua.
- Chấm: `uv run pytest tracks/data-analyst/modules/13-business-financial-analytics/exercises/basic/02-order-unit-economics -q`

In [ ]:
# explore: tìm gốc repo, tải bộ dữ liệu nếu chưa có, đọc sheet 2010-2011 (lần đầu khoảng 10 giây, sau đó đọc cache)
import os
import sys
import tempfile
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure
import duckdb
import pandas as pd

XLSX = ensure("online-retail-ii") / "online_retail_II.xlsx"


def read_sheet(sheet: str) -> pd.DataFrame:
    """Một sheet, đúng dữ liệu gốc. Lần đầu đọc xlsx (DuckDB khoảng 7 giây; máy không cài được extension excel
    thì pandas 30-90 giây), rồi cache parquet trong thư mục tạm (khoá theo mtime + size); lần sau khoảng 1 giây."""
    stat = XLSX.stat()
    cache = (Path(tempfile.gettempdir()) / "da-cache"
             / f"online_retail_II-{sheet.replace(' ', '_')}-{stat.st_mtime_ns}-{stat.st_size}.parquet")
    if cache.exists():
        return pd.read_parquet(cache)
    path = str(XLSX).replace("'", "''")
    try:
        df = duckdb.sql(f"""
            SELECT Invoice, StockCode, Description, CAST(CAST(Quantity AS DOUBLE) AS BIGINT) AS Quantity,
                   TIMESTAMP '1899-12-30' + to_seconds(CAST(round(CAST(InvoiceDate AS DOUBLE) * 86400) AS BIGINT))
                       AS InvoiceDate,
                   CAST(Price AS DOUBLE) AS Price, CAST("Customer ID" AS DOUBLE) AS "Customer ID", Country
            FROM read_xlsx('{path}', sheet = '{sheet}', all_varchar = true)""").df()
    except (duckdb.CatalogException, duckdb.IOException, duckdb.HTTPException) as exc:
        if "extension" not in str(exc).lower():
            raise
        df = pd.read_excel(XLSX, sheet_name=sheet)          # máy offline: không cài được extension excel
    df = df.assign(Invoice=df["Invoice"].astype(str), StockCode=df["StockCode"].astype(str),
                   Description=df["Description"].astype("string"))
    cache.parent.mkdir(parents=True, exist_ok=True)
    tmp = cache.with_name(f"{cache.name}.{os.getpid()}.tmp")
    df.to_parquet(tmp, index=False)
    os.replace(tmp, cache)
    return df

tx = read_sheet("Year 2010-2011")
print(tx.shape, tx.InvoiceDate.min(), "→", tx.InvoiceDate.max())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ASSUMPTIONS = {"cogs_rate": 0.60, "payment_fee_rate": 0.02, "cost_per_order": 8.0, "cost_per_line": 0.25}

## 1. Bảng đơn hàng

`order_table(tx) -> pd.DataFrame`: một dòng mỗi hoá đơn hợp lệ (trên các dòng hợp lệ), index `Invoice` kiểu `str`
tăng dần, bốn cột theo thứ tự:

| Cột | Định nghĩa |
| --- | --- |
| `revenue` | tổng `Quantity * Price` của hoá đơn |
| `lines` | số dòng hợp lệ của hoá đơn |
| `country` | `Country` của dòng đầu tiên |
| `customer_id` | `Customer ID` của dòng đầu tiên (`NaN` với khách vãng lai) |

Giữ mọi dòng hợp lệ, kể cả dòng phí như `POST` (cước khách trả): đó là doanh thu theo định nghĩa chung của track.

In [ ]:
def order_table(tx: pd.DataFrame) -> pd.DataFrame:
    """Một dòng mỗi hoá đơn hợp lệ: revenue, lines, country, customer_id; index Invoice (str)."""
    # TODO: lọc dòng hợp lệ (Invoice ép str), revenue dòng, groupby Invoice
    raise NotImplementedError

## 2. Unit economics và P&L thu nhỏ

`unit_economics(orders, assumptions, price_change=0.0) -> pd.DataFrame`: bản sao `orders` thêm năm cột, khi giá
mọi mặt hàng đổi `price_change` (0.05 là tăng 5%) và số lượng giữ nguyên:

| Cột | Công thức |
| --- | --- |
| `revenue` | `revenue` gốc × (1 + `price_change`) |
| `cogs` | `cogs_rate` × `revenue` **gốc** (giá vốn theo số lượng hàng, không đổi theo giá bán) |
| `fees` | `payment_fee_rate` × `revenue` mới |
| `fulfilment` | `cost_per_order` + `cost_per_line` × `lines` |
| `contribution` | `revenue − cogs − fees − fulfilment` |

`pnl(ue, volume_change=0.0) -> pd.Series` (tên `pnl`): tổng của bảng `ue` khi số đơn đổi `volume_change` (−0.1 là
mất 10% đơn, mọi đơn co đều nhau, nên mọi tổng nhân `1 + volume_change`). Index theo thứ tự:
`revenue`, `cogs`, `gross_profit` (= revenue − cogs), `gross_margin` (= gross_profit / revenue), `fees`,
`fulfilment`, `contribution`, `contribution_margin` (= contribution / revenue), `orders` (số đơn sau khi đổi),
`contribution_per_order`.

`breakeven_order_value(lines, assumptions) -> float`: giá trị đơn (ở giá hiện tại) mà contribution bằng 0 với
`lines` dòng hàng.

In [ ]:
def unit_economics(orders: pd.DataFrame, assumptions: dict, price_change: float = 0.0) -> pd.DataFrame:
    """Thêm revenue (giá mới), cogs, fees, fulfilment, contribution cho từng đơn."""
    # TODO: chú ý cogs tính trên revenue GỐC, fees trên revenue MỚI
    raise NotImplementedError


def pnl(ue: pd.DataFrame, volume_change: float = 0.0) -> pd.Series:
    """P&L thu nhỏ: tổng các khoản, margin, số đơn, contribution mỗi đơn."""
    # TODO
    raise NotImplementedError


def breakeven_order_value(lines: float, assumptions: dict) -> float:
    """Giá trị đơn để contribution = 0 với `lines` dòng hàng."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: P&L thu nhỏ cả năm và phân phối contribution theo đơn
orders = order_table(tx)
ue = unit_economics(orders, ASSUMPTIONS)
print(pnl(ue).round(3).to_string())
neg = ue[ue.contribution < 0]
print(f"\n{len(orders):,} đơn; {len(neg):,} đơn contribution âm ({len(neg) / len(orders):.1%}), "
      f"tổng lỗ {neg.contribution.sum():,.0f} GBP")
print("giá trị đơn: median", round(orders.revenue.median(), 2), "| số dòng: median", orders.lines.median())
print("điểm hoà vốn với số dòng median:", round(breakeven_order_value(orders.lines.median(), ASSUMPTIONS), 2), "GBP")
orders.revenue.describe(percentiles=[.1, .25, .5, .75, .9]).round(2)

In [ ]:
# explore: contribution theo nhóm giá trị đơn và theo thị trường
bins = [0, 25, 50, 100, 250, 500, 1000, np.inf]
ue["band"] = pd.cut(ue.revenue, bins, right=False)
ue["market"] = np.where(ue.country == "United Kingdom", "UK", "Export")
by_band = ue.groupby("band", observed=True).agg(orders=("revenue", "size"), revenue=("revenue", "sum"),
                                                contribution=("contribution", "sum"))
by_band["cm_per_order"] = by_band.contribution / by_band.orders
print(by_band.round(1))
print(ue.groupby("market").agg(orders=("revenue", "size"), revenue=("revenue", "sum"),
                               contribution=("contribution", "sum"),
                               cm_per_order=("contribution", "mean")).round(1))

In [ ]:
# explore: chart - contribution mỗi đơn theo nhóm giá trị đơn
fig, ax = plt.subplots(figsize=(8, 3.5))
colors = ["#e45756" if v < 0 else "#4c78a8" for v in by_band.cm_per_order]
ax.bar([str(b) for b in by_band.index], by_band.cm_per_order, color=colors)
ax.axhline(0, color="black", lw=0.8)
ax.set_ylabel("GBP / đơn")
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("Đơn dưới 25 GBP lỗ trung bình mỗi đơn (giả định chi phí ở đầu notebook)", loc="left")
plt.xticks(rotation=20)
plt.show()

## 3. Bảng kịch bản (what-if)

`scenario_table(orders, assumptions, scenarios) -> pd.DataFrame`: `scenarios` là dict tên → dict thay đổi, có thể
gồm bất kỳ khoá nào của `assumptions` (ghi đè giả định), và:

- `price_change` (mặc định 0): đổi giá như ở `unit_economics`;
- `min_order_value` (mặc định không có): bỏ các đơn có `revenue` (sau khi đổi giá) **nhỏ hơn** ngưỡng, coi như mất đơn;
- `volume_change` (mặc định 0): số đơn còn lại đổi theo tỉ lệ, như ở `pnl`.

Kết quả: dòng đầu tên `base` (không đổi gì), sau đó các kịch bản theo thứ tự trong dict. Cột theo thứ tự `revenue`,
`contribution`, `contribution_margin`, `orders`, `delta_contribution` (contribution trừ contribution của `base`).

`breakeven_volume_change(orders, assumptions, price_change) -> float`: tỉ lệ đổi số đơn (âm) mà khi giá đổi
`price_change`, tổng contribution **bằng** contribution hiện tại. Gợi ý: contribution tỉ lệ thuận với số đơn.

In [ ]:
def scenario_table(orders: pd.DataFrame, assumptions: dict, scenarios: dict) -> pd.DataFrame:
    """Dòng base + mỗi kịch bản: revenue, contribution, contribution_margin, orders, delta_contribution."""
    # TODO: với mỗi kịch bản: giả định ghi đè -> unit_economics -> bỏ đơn dưới ngưỡng -> pnl(volume_change)
    raise NotImplementedError


def breakeven_volume_change(orders: pd.DataFrame, assumptions: dict, price_change: float) -> float:
    """Tỉ lệ đổi số đơn để contribution ở giá mới bằng contribution hiện tại."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: bảng kịch bản cho CFO
scenarios = {
    "Giá +5%, mất 8% đơn": {"price_change": 0.05, "volume_change": -0.08},
    "Đơn tối thiểu 25 GBP": {"min_order_value": 25},
    "Cước giao vận +25%": {"cost_per_order": 10.0},
    "Nhà cung cấp tăng giá 3 điểm": {"cogs_rate": 0.63},
}
table = scenario_table(orders, ASSUMPTIONS, scenarios)
print(table.round({"revenue": 0, "contribution": 0, "contribution_margin": 4, "orders": 0, "delta_contribution": 0}))
for p in [0.02, 0.05, 0.10]:
    print(f"giá +{p:.0%}: được mất tới {-breakeven_volume_change(orders, ASSUMPTIONS, p):.1%} số đơn")

In [ ]:
# explore: độ nhạy (tornado) - mỗi giả định ±20%, các giả định khác giữ nguyên
base_c = pnl(unit_economics(orders, ASSUMPTIONS))["contribution"]
rows = []
for key, value in ASSUMPTIONS.items():
    lo = pnl(unit_economics(orders, {**ASSUMPTIONS, key: value * 0.8}))["contribution"] - base_c
    hi = pnl(unit_economics(orders, {**ASSUMPTIONS, key: value * 1.2}))["contribution"] - base_c
    rows.append((key, lo, hi))
tornado = pd.DataFrame(rows, columns=["assumption", "minus20", "plus20"]).set_index("assumption")
tornado = tornado.reindex(tornado.abs().max(axis=1).sort_values().index)
print(tornado.round(0))
fig, ax = plt.subplots(figsize=(8, 3))
ax.barh(tornado.index, tornado.minus20 / 1e3, color="#4c78a8", label="giả định −20%")
ax.barh(tornado.index, tornado.plus20 / 1e3, color="#e45756", label="giả định +20%")
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("thay đổi contribution (nghìn GBP)")
ax.legend(frameon=False, loc="lower right")
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("Kết luận nhạy nhất với giả định giá vốn", loc="left")
plt.show()

## 4. Đọc kết quả, rồi làm trắc nghiệm

1. Bao nhiêu phần trăm đơn có contribution âm, chúng nằm ở nhóm giá trị nào? Lỗ bao nhiêu so với tổng contribution?
2. Đặt đơn tối thiểu 25 GBP có đáng không, nếu khách bị chặn **bỏ đi** (kịch bản trong bảng)? Nếu một nửa trong số
   họ mua thêm cho đủ 25 GBP thì sao?
3. Tăng giá 5% được phép mất tối đa bao nhiêu phần trăm đơn? Viết cho CFO một câu, ghi rõ con số phụ thuộc giả
   định nào nhất (xem tornado).
4. Export và UK khác nhau thế nào về contribution mỗi đơn? Giả định "8 GBP mỗi đơn" có hợp lý cho đơn xuất khẩu không?

Xong thì mở app, làm phần **trắc nghiệm** của bài.